In [20]:
import os
import numpy as np
import networkx as nx

from src.OrganoidMesh import OrganoidMesh
from src.cell_graph_functions import *
from src.nuclei_to_mesh_projection import *
from src.mesh_analysis import *
from src.crypt_extraction import *
from src.organoid_plotting import *

In [21]:
# ---------------------------------------------------------------------------
# Configuration / paths
# ---------------------------------------------------------------------------

data_dir = '../NicoleData/20250929/fractal_output'

# --- path to your cell nucleus table ---
CELLS_CSV = "../NicoleData/features_csv/features/cell_types_class.csv"   # <-- adjust

timepoint = "day4p5"
zarr_name = "r0.zarr"
well = "A06"
round_name = "0_fused_zillum_registered"
organoid_id = 31 #19 # example


mesh_path = f"{data_dir}/{timepoint}/{zarr_name}/{well[0]}/{well[1:]}/{round_name}/meshes/nnorg_linked_multi_annotated_class/{organoid_id}.vtp"
cells_df = pd.read_csv(CELLS_CSV)

### Project nuclei to mesh and tessellate

In [22]:
organoid_id = os.path.splitext(os.path.basename(mesh_path))[0]
label_uid = f"{timepoint}_{well}_{organoid_id}"

nuclei_df_org = cells_df[cells_df["label_uid"] == label_uid].copy()

# 1) load membrane mesh as an OrganoidMesh
mesh = OrganoidMesh(mesh_path)

# 2) extract per-cell attributes
nuclei_xyz, markers_bin = extract_cell_attributes(nuclei_df_org)

_, nuclei_xyz = center_and_rescale_mesh(mesh, nuclei_xyz)
markers_bin = filter_lgr5_coexpression(markers_bin) 

# 3) project nuclei -> mesh vertices (using geometry from the mesh object)
proj_vertex_ids, proj_points = project_nuclei_to_mesh(
    nuclei_xyz,
    mesh,
    resolve_duplicates=True,
)

# store eigen-decomposition for geodesics (if needed by compute_geodesics)
mesh._eig_decomp()

# 4) geodesic distances + Voronoi assignment
dist_mat, vertex_owner = compute_geodesic_voronoi(mesh, proj_vertex_ids)

# 5) build graph from Voronoi partition (now pass mesh, not mesh_f)
G = build_cell_graph_from_voronoi(
    nuclei_xyz,
    markers_bin,
    mesh,            
    vertex_owner,
    proj_vertex_ids,
    proj_points,
)

Duplicate projections: 9 vertices with duplicates, 9 duplicate cell pairs.


heat sources: 100%|██████████| 1030/1030 [00:04<00:00, 237.08it/s]


In [23]:
t_hks = [1.0, 2.0, 4.0, 8.0, 25.0] 
HKS = compute_hks(mesh, t=t_hks, coeffs=False)   # (V, T)


# store times as graph-level metadata (no need to duplicate per node)
add_vertex_field_to_graph(G, HKS, "hks")

# Compute encoing
vocab = np.load('./sim/vocab.npz', allow_pickle=True)

encoding, _, _, _ = compute_vocabulary_encoding(vocab, mesh)
add_vertex_field_to_graph(G, encoding, "vocab_encoding")

/home/fmoller/miniforge3/envs/SphericalHarmonics/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning:

Trying to unpickle estimator Normalizer from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations



### Segment crypts and villus from vocab

In [24]:
crypt_vocab_idx = [1, 2, 7]

crypts, villus = segment_organoid_from_vocab(
    G,
    crypt_vocab_idx=crypt_vocab_idx,
    crypt_thresh=0.01,
    min_crypt_region_size=10,
)

In [25]:

fig = plot_organoid_graph(proj_points, G, HKS[:,0], node_size=2)
add_region_overlays(fig, proj_points, crypts, colorscale='YlOrRd')
add_region_overlays(fig, proj_points, villus, colorscale='viridis')

fig.show()

### Extract crypt composition

In [ ]:
def _to_patch_list(x):
    """Normalize to list[set[int]]: None -> [], set -> [set], list/tuple -> list of sets."""
    if x is None:
        return []
    if isinstance(x, set):
        return [x]
    if isinstance(x, (list, tuple)):
        return [set(p) for p in x if p is not None and len(p) > 0]
    # anything else: ignore
    return []


def get_crypt_neck_boundary_vertices(mesh, vertex_owner, crypt_cells, neck_cells):
    """
    Identify a *single-layer* ring of mesh vertices lying on the perimeter
    between a crypt and its neck/villus, biased to the neck side.

    Parameters
    ----------
    mesh : object
        Mesh with attributes:
            - v: (V, 3) array of vertex coordinates
            - f: (F, 3) array of triangle vertex indices (int)
    vertex_owner : (V,) array-like of int
        For each vertex, the index of the closest / owning cell.
    crypt_cells : set or array-like of int
        Cell indices belonging to the crypt of interest.
    neck_cells : set or array-like of int
        Cell indices belonging to the neck/villus region touching this crypt.

    Returns
    -------
    boundary_vertices : np.ndarray of int, shape (K,)
        Sorted array of vertex indices that:
        - are owned by neck cells, and
        - have at least one edge neighbor owned by a crypt cell.
    """

    vertex_owner = np.asarray(vertex_owner, dtype=np.int64)
    crypt_cells  = np.asarray(list(crypt_cells), dtype=np.int64)
    neck_cells   = np.asarray(list(neck_cells), dtype=np.int64)

    V = vertex_owner.shape[0]

    # --- 1) Label vertices by region ---
    is_crypt_vertex = np.isin(vertex_owner, crypt_cells)
    is_neck_vertex  = np.isin(vertex_owner, neck_cells)

    # --- 2) Build vertex adjacency from faces ---
    neighbors = [[] for _ in range(V)]
    faces = np.asarray(mesh.f, dtype=np.int64)

    for tri in faces:
        a, b, c = tri
        neighbors[a].extend([b, c])
        neighbors[b].extend([a, c])
        neighbors[c].extend([a, b])

    # make neighbors unique
    neighbors = [np.unique(n) for n in neighbors]

    # --- 3) Neck-side boundary: neck vertices with at least one crypt neighbor ---
    boundary_mask = np.zeros(V, dtype=bool)

    for v in range(V):
        if not is_neck_vertex[v]:
            continue
        # does this neck vertex see any crypt vertex via an edge?
        if np.any(is_crypt_vertex[neighbors[v]]):
            boundary_mask[v] = True

    boundary_vertices = np.nonzero(boundary_mask)[0]
    return np.sort(boundary_vertices)





crypt_patches = _to_patch_list(crypts)
villus_patches = _to_patch_list(villus)

# union of all villus patches into a single set of cell IDs
if villus_patches:
    neck_cells_all = set().union(*villus_patches)
else:
    neck_cells_all = set()


for crypt_cells in crypt_patches:

    boundary_vertex_ids = get_crypt_neck_boundary_vertices(
        mesh, vertex_owner, crypt_cells, neck_cells_all
    )


    fig = go.Figure()
    fig.add_trace(
        go.Mesh3d(
            x=mesh.v[:,0], y=mesh.v[:,1], z=mesh.v[:,2],
            i=mesh.f[:,0], j=mesh.f[:,1], k=mesh.f[:,2],
            opacity=0.5,
            name="Mesh"
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=mesh.v[boundary_vertex_ids,0],
            y=mesh.v[boundary_vertex_ids,1],
            z=mesh.v[boundary_vertex_ids,2],
            mode="markers",
            marker=dict(size=4, color="red"),
            name="Crypt–neck perimeter"
        )
    )
    fig.show()


(1030, 18001)


In [ ]:
import numpy as np
from mesh_analysis import compute_geodesics  # Heat-method geodesics on vertices

def normalized_bottom_distance_heat_neck(
    mesh,
    dist_mat,
    bottom_cell_idx,
    boundary_vertex_ids,
    vertex_owner=None,
    t=None,
):
    """
    Compute normalized distance to the crypt bottom using a vertex-wise scheme:

      1. Extract distances from the crypt-bottom cell to all vertices
         from dist_mat[bottom_cell_idx, :].
      2. Compute geodesic distances using the neck/boundary vertices as sources
         via the Heat Method on the mesh.
      3. For each vertex, find the closest neck vertex (w.r.t. these neck→vertex
         distances).
      4. For each vertex, define L(v) as the distance from crypt bottom to that
         closest neck vertex.
      5. For each vertex, return d_norm(v) = dist_bottom(v) / L(v).

    Optionally also returns per-cell normalized distances by averaging over
    vertices owned by each cell.

    Parameters
    ----------
    mesh : OrganoidMesh
        Must provide v, f, laplacian, mass_matrix (for compute_geodesics).
    dist_mat : (N_cells, V) ndarray
        Geodesic distances from each cell-center vertex to all vertices.
    bottom_cell_idx : int
        Index of the bottom cell in this crypt (row index into dist_mat).
    boundary_vertex_ids : (K,) array-like of int
        Vertex indices on the crypt–neck perimeter (neck/villus side).
    vertex_owner : (V,) array-like of int, optional
        For each vertex, the index of the closest/owning cell. If provided,
        per-cell normalized distances will be computed.
    t : float, optional
        Diffusion time for compute_geodesics. If None, the function's default
        heuristic is used.

    Returns
    -------
    dnorm_vertices : (V,) ndarray
        Normalized distance-to-bottom for each vertex.
    dnorm_cells : (N_cells,) ndarray or None
        Normalized distance per cell (mean over its owned vertices) if
        vertex_owner is provided, otherwise None.
    """

    dist_mat = np.asarray(dist_mat, dtype=float)
    boundary_vertex_ids = np.asarray(boundary_vertex_ids, dtype=int)

    # --- 1) distances from crypt-bottom cell to all vertices ---
    dist_bottom = dist_mat[bottom_cell_idx, :]  # shape (V,)

    # --- 2) geodesics from neck/boundary vertices as sources ---
    #    D_neck: (K, V) distances, row k = boundary_vertex_ids[k] → all vertices
    D_neck = compute_geodesics(mesh, t=t, sources=boundary_vertex_ids)

    # --- 3) closest neck vertex for each vertex ---
    closest_neck_idx = np.argmin(D_neck, axis=0)              # shape (V,)
    closest_neck_vertex = boundary_vertex_ids[closest_neck_idx]  # (V,)

    # --- 4) L(v) = dist(bottom → closest_neck_vertex(v)) ---
    L = dist_bottom[closest_neck_vertex]                      # (V,)
    L = np.maximum(L, 1e-8)  # numerical safety

    # --- 5) normalized distances ---
    dnorm_vertices = dist_bottom / L

    # Optional: aggregate back to cells
    dnorm_cells = None
    if vertex_owner is not None:
        vertex_owner = np.asarray(vertex_owner, dtype=int)
        n_cells = dist_mat.shape[0]
        dnorm_cells = np.zeros(n_cells, dtype=float)

        for c in range(n_cells):
            mask = (vertex_owner == c)
            if not np.any(mask):
                dnorm_cells[c] = np.nan
            else:
                dnorm_cells[c] = np.mean(dnorm_vertices[mask])

    return dnorm_vertices, dnorm_cells

